In [1]:
import pandas as pd 
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

In [134]:
relevant_cols = ["Gene", "Sample ID", "Cancer Type Detailed", "Mutation Type", "Variant Type", "HGVSc", "MS", 
        "Protein Change", "Functional Impact"]
early_age = 50

data_path = os.path.abspath("../data/tcga_data2.csv")
raw_df = pd.read_csv(data_path, sep = '\t', low_memory = False)
raw_df.head(1)

,Gene,Study of Origin,Sample ID,Cancer Type,Cancer Type Detailed,Protein Change,Annotation,Custom Driver,Custom Driver Tiers,Functional Impact,...,Tumor Type,Used in Genomic Analysis,Vascular invasion indicator,Vessel Invasion,Vial number,Patient's Vital Status,Patient Weight,WGD,Winter Hypoxia Score,Year of Diagnosis
0,APC,"Colorectal Adenocarcinoma (TCGA, Firehose Legacy)",TCGA-AA-A010-01,Colorectal Cancer,Colon Adenocarcinoma,A2D,"OncoKB: Unknown, level NA, resistance NA;reVUE...",NaN,NaN,MutationAssessor: NA;SIFT: impact: deleterious...,...,NaN,NaN,NO,NaN,A,NaN,NaN,NaN,NaN,NaN


In [60]:
def groupby_age(df):
    cols = np.array(df.columns.tolist())
    pattern = re.compile(r'\bage\b', flags = re.IGNORECASE)
    age_cols = np.array([col for col in cols if pattern.search(col) and "Diagnosis" in col])
    df.loc[:, 'Age'] = df[age_cols].bfill(axis = 1).iloc[:, 0]
    df = df.drop(columns = age_cols)
    return df

In [35]:
def columns(df, relevant_cols):
    necessary_cols = ['Sample ID','HGVSc']
    if set(relevant_cols).intersection(necessary_cols) == set(necessary_cols):
        good_cols = np.concatenate((relevant_cols, ['Age']))
        new_df = df[good_cols].copy()
        return new_df
    else:
        raise ValueError("Relevant columns must contain sample ID and HGVSc")

In [59]:
def strip_strings(df):
    df.columns = df.columns.str.strip()
    return df

In [12]:
def remove_duplicates(df):
    df = df.dropna(subset = ["Age"], axis = 0)
    df = df.drop_duplicates(subset = ["Sample ID", "HGVSc"], keep = "first")
    return df

In [133]:
def early_onset(df, early_age):
    df = df.copy()
    df["Early Onset"] = df["Age"] < early_age
    return df

In [136]:
pipeline = (raw_df
    .pipe(groupby_age)
    .pipe(columns, relevant_cols = relevant_cols)
    .pipe(remove_duplicates)
    .pipe(early_onset, early_age = early_age)
           )

pipeline.head(1)

,Gene,Sample ID,Cancer Type Detailed,Mutation Type,Variant Type,HGVSc,MS,Protein Change,Functional Impact,Age,Early Onset
0,APC,TCGA-AA-A010-01,Colon Adenocarcinoma,Missense_Mutation,SNP,ENST00000257430.4:c.5C>A,Somatic,A2D,MutationAssessor: NA;SIFT: impact: deleterious...,46.0,True


---
Extra functions

---

In [116]:
def early_onset_df(df, early_age):
    df = df.copy()
    df["Early Onset"] = df["Age"] < early_age
    df = df[df["Early Onset"] == True]
    return df

In [124]:
def find_gene(df, gene):
    return df[df["HGVSc"] == gene]

---
Trying to find a null cohort - Lung Cancer Data

---

In [135]:
data_path = os.path.abspath("../data/lung_data.csv")
lung_df = pd.read_csv(data_path, low_memory = False)
new_pipeline = (lung_df
    .pipe(strip_strings)
    .pipe(groupby_age)
    .pipe(columns, relevant_cols = relevant_cols)
    .pipe(remove_duplicates)
    .pipe(early_onset, early_age = early_age)
           )
new_pipeline.head(1)

,Gene,Sample ID,Cancer Type Detailed,Mutation Type,Variant Type,HGVSc,MS,Protein Change,Functional Impact,Age,Early Onset
7,APC,LUAD-NYU408,Lung Adenocarcinoma,Splice_Site,SNP,ENST00000257430.4:c.1744-2A>G,Unknown,X582_splice,MutationAssessor: NA;SIFT: NA;Polyphen-2: NA;A...,71.0,False


In [122]:
crc_df = early_onset_df(pipeline, 50)
lung_df = early_onset_df(new_pipeline, 55)
lung_df.shape

(35, 11)

--- 
Checking to see if they contain the mutation

---

In [132]:
find_gene(new_pipeline, "ENST00000257430.4:c.835-8A>G").shape 
# find_gene(pipeline, "ENST00000257430.4:c.835-8A>G").shape

(0, 10)